# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the "FAIRˆ²" dataset package using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All entities are referenced by their `@id` fields, ensuring reproducibility and clarity.

### Dataset Source
This dataset is defined by a Croissant schema and accessible via the URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a single object, not a dict
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.
We will enumerate the available record sets and, for each, list its fields and columns (with their `@id`s).

In [ ]:
# List available record sets and their `@id`
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    # Fallback for `recordSet` attribute if present as per Croissant 1.0
    record_sets = getattr(metadata, 'recordSet', [])

# Display record set IDs and field details
record_set_ids = []
for rs in record_sets:
    print(f"Record Set: {getattr(rs, '@id', 'N/A')}")
    record_set_ids.append(getattr(rs, '@id', None))
    # List fields
    if hasattr(rs, 'fields'):
        fields = rs.fields
    else:
        fields = getattr(rs, 'field', [])
    for f in fields:
        print(f"  Field: {getattr(f, '@id', 'N/A')}, Name: {getattr(f, 'name', 'N/A')}")
        # List columns for this field
        if hasattr(f, 'columns'):
            columns = f.columns
        else:
            columns = getattr(f, 'column', [])
        for c in columns:
            print(f"    Column: {getattr(c, '@id', 'N/A')}, Name: {getattr(c, 'name', 'N/A')}")
    print('---')
if not record_set_ids:
    print("No record sets found in the metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
All entities (record sets, fields and columns) are referenced by their `@id` values as listed above.

In [ ]:
# Prepare a list of record set IDs
record_sets_available = record_set_ids  # from previous cell

dataframes = dict()

for record_set_id in record_sets_available:
    if record_set_id:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
        else:
            print(f"No records loaded for {record_set_id}.")
    else:
        print("Invalid record set ID or empty value.")

# Display structure for the first non-empty DataFrame
first_df_id = None
for rid, df in dataframes.items():
    if not df.empty:
        first_df_id = rid
        break
if first_df_id:
    print(f"First record set DataFrame columns ({first_df_id}): ", dataframes[first_df_id].columns.tolist())
    display(dataframes[first_df_id].head())
else:
    print("No dataframes with records to display.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering on a numeric field, normalizing numeric columns, and grouping. All references are made using `@id` where applicable.

> **Note:** For demonstration, we select the first available numeric field from the first DataFrame. Update the variable assignments as needed for your domain use.

In [ ]:
from pandas.api.types import is_numeric_dtype

# Choose a DataFrame for EDA
if not dataframes:
    raise ValueError('No dataframes available for analysis.')

df_id = first_df_id
df = dataframes[df_id]

# Identify numeric field and grouping field by trying the columns
numeric_field = None
group_field = None
for col in df.columns:
    if is_numeric_dtype(df[col]):
        numeric_field = col
        break
for col in df.columns:
    # Take a non-numeric or categorical field for grouping
    if not is_numeric_dtype(df[col]):
        group_field = col
        break

if numeric_field is not None:
    threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    filtered_df = filtered_df.copy()  # Avoid SettingWithCopyWarning
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA in the selected dataframe.")

## 5. Visualization
Visualize data distributions or relationships between fields.

> For demonstration, we plot the distribution of the numeric field and a box plot grouped by the chosen grouping field (if applicable).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    fig, axes = plt.subplots(1, 2 if group_field else 1, figsize=(12, 4))

    # Histogram
    ax1 = axes[0] if group_field else axes
    sns.histplot(df[numeric_field].dropna(), bins=10, kde=True, ax=ax1)
    ax1.set_title(f"Distribution of {numeric_field}")

    # Boxplot by group (if possible)
    if group_field and group_field in df.columns:
        ax2 = axes[1]
        sns.boxplot(x=group_field, y=numeric_field, data=df, ax=ax2)
        ax2.set_title(f"{numeric_field} by {group_field}")
        plt.setp(ax2.get_xticklabels(), rotation=30, ha='right')

    plt.tight_layout()
    plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR⁲ dataset using the `mlcroissant` library. We showed how to enumerate available record sets, load them by their `@id`, and carry out a basic exploratory data analysis pipeline using exclusively Croissant-compliant referencing. For deeper analysis, refer to individual field and column `@id`s as per the Croissant schema documentation and adjust visualizations or analyses as needed for domain questions.